In [2]:
import pandas as pd
import geopandas as gpd
import altair as alt
from pathlib import Path

In [159]:
MERGED_SF_TRACTS_SHP = (
    Path.cwd().parent / "clean-data/merged_sf_shapefiles/merged_sf_tracts.shp"
)

MERGED = Path.cwd().parent / "clean-data/merged_data.csv"

In [154]:
def create_tract_map(start_date: str, end_date: str, col_name: str):
    """
    Add docstring
    """
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    merged_df = pd.read_csv(MERGED)
    tracts_gdf = gpd.read_file(MERGED_SF_TRACTS_SHP)

    merged_df["tract"] = merged_df["tract"].astype(str).str.zfill(11)

    # Filter
    filtered_df = (
        merged_df[(merged_df["date"] >= start_date) & (merged_df["date"] <= end_date)]
        .drop(columns=["date"])
        .groupby("tract")
        .mean()
        .reset_index()
    )

    # Merge filtered, aggregated dataframe with GeoDataFrame
    merged_tracts_gdf = tracts_gdf.merge(filtered_df, left_on="GEOID", right_on="tract", how="left")


    tooltips = [
        alt.Tooltip("GEOID:N", title="Tract ID"),
        alt.Tooltip("population:Q", title="Population"),
        alt.Tooltip("median_rent:Q", format=",.0f", title="Median rent (per month)"),
        alt.Tooltip("eviction_rate:Q", format=".3%", title="Average eviction rate"),
        alt.Tooltip("estimate:Q", format=",.0f", title="Average homeless population estimate"),
        alt.Tooltip("311_calls:Q", format=",.0f", title="Average monthly citizen-reported encampments")
    ]

    if col_name in ["tents", "structures", "vehicles"]:
        tooltips.append(alt.Tooltip(f"{col_name}:Q", format=",.0f", title=f"Average {METRIC_NAMES[col_name].lower()}"))

    # Create base map for tracts with no data
    background = (
        alt.Chart(tracts_gdf)
        .mark_geoshape(fill="lightgray", stroke="white")
        .project("albersUsa")
    )

    # Build chart
    chart = (
        alt.Chart(merged_tracts_gdf)
        .mark_geoshape()
        .encode(
            color=alt.Color(f"{col_name}:Q", title={METRIC_NAMES[col_name]}),
            tooltip=tooltips
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, ("tract"), ["metric"]),
        )
        .project(type="albersUsa")
        .properties(
            title=f"Average {METRIC_NAMES[col_name].lower()} in SF tracts ({start_date} to {end_date})"
        )
        .interactive()
        )

    return background + chart

In [ ]:
def create_tract_map(start_date: str, end_date: str, col_name: str):
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    merged_df = pd.read_csv(MERGED)
    tracts_gdf = gpd.read_file(MERGED_SF_TRACTS_SHP)

    merged_df["tract"] = merged_df["tract"].astype(str).str.zfill(11)

    # merged_df["date"] = pd.to_datetime(merged_df)
    # start_dt = pd.to_datetime(start_date)
    # end_dt = pd.to_datetime(end_date)

    filtered_df = (
        merged_df[(merged_df["date"] >= start_date) & (merged_df["date"] <= end_date)]
        .drop(columns=["date"])
        .groupby("tract")
        .mean()
        .reset_index()
    )

    # Merge aggregated dataframe with GeoDataFrame
    merged_tracts_gdf = tracts_gdf.merge(filtered_df, left_on="GEOID", right_on="tract", how="left")

    # Create tooltips, and add last tooltip only if selected column name is not one of the existing tooltips
    tooltips = [
        alt.Tooltip("GEOID:N", title="Tract ID"),
        alt.Tooltip("population:Q", title="Population"),
        alt.Tooltip("median_rent:Q", format=",.0f", title="Median rent (per month)"),
        alt.Tooltip("eviction_rate:Q", format=".3%", title="Average eviction rate"),
        alt.Tooltip("estimate:Q", format=",.0f", title="Average homeless population estimate"),
        alt.Tooltip("311_calls:Q", format=",.0f", title="Average monthly citizen-reported encampments")
    ]

    if col_name in ["tents", "structures", "vehicles"]:
        tooltips.append(alt.Tooltip(f"{col_name}:Q", format=",.0f", title=f"Average {METRIC_NAMES[col_name].lower()}"))

    # Create base map for tracts with no data
    background = (
        alt.Chart(tracts_gdf)
        .mark_geoshape(fill="lightgray", stroke="white")
        .project("albersUsa")
    )

    # Build interactive choropleth map
    chart = (
        alt.Chart(merged_tracts_gdf)
        .mark_geoshape(stroke="black", strokeWidth=0.5)
        .encode(
            color=alt.Color(
                f"{col_name}:Q",
                title=METRIC_NAMES[col_name],
                legend=alt.Legend(
                    orient="right",
                    direction="vertical",
                    labelAlign="left",
                    titlePadding=5,
                    offset=10,
                ),
            ),
            tooltip=tooltips
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, "tract", ["metric"]),
        )
        .project(type="mercator", scale=120000, center=[-122.43, 37.77])
        .properties(width="container", height=550)
        .configure_view(stroke=None)
        .interactive()
    )

    return chart.resolve_scale(color="independent")

In [165]:
create_tract_map("2020-01", "2024-12", "estimate")

ValueError: to assemble mappings requires at least that [year, month, day] be specified: [day,month,year] is missing